# _Bag of words_

In [1]:
import pandas as pd
import re
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.model_selection import cross_val_score

In [2]:
train = pd.read_csv("data/labeledTrainData.tsv", header=0,
                    delimiter="\t", quoting=3)

test = pd.read_csv("data/testData.tsv", header=0,
                   delimiter="\t", quoting=3)

In [3]:
def handle_text(raw_text):
    soup = BeautifulSoup(raw_text)
    without_html = soup.get_text()

    letters_only = re.sub('[^a-zA-Z]', ' ', without_html)

    lower_case = letters_only.lower()
    return lower_case

In [4]:
vectorizer = CountVectorizer(analyzer = "word",
                             tokenizer = None,
                             preprocessor = handle_text,
                             stop_words = 'english',
                             max_features = 5000)

train_data_features = vectorizer.fit_transform(train['review'])
joblib.dump(vectorizer, 'models/bow')

train_data_features = train_data_features.toarray()

In [5]:
model = RandomForestClassifier(n_estimators=100)
model = model.fit(train_data_features, train['sentiment'])

In [6]:
test_data_features = vectorizer.transform(test['review'])
test_data_features = test_data_features.toarray()

result = model.predict(test_data_features)

output = pd.DataFrame(data = {"id":test["id"], "sentiment":result})
output.to_csv("results/Bag_of_Words_model.csv", index = False, quoting=3)

# _Metrics_

In [7]:
cv_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

accuracy_scores = cross_val_score(cv_model, train_data_features, train['sentiment'], cv=5, scoring='accuracy')
print(f"Кросс-валидация Accuracy: {accuracy_scores.mean() * 100:.2f}% (разброс: +/- {accuracy_scores.std() * 100:.2f}%)")

roc_auc_scores = cross_val_score(cv_model, train_data_features, train['sentiment'], cv=5, scoring='roc_auc')
print(f"Кросс-валидация ROC AUC:  {roc_auc_scores.mean() * 100:.2f}%")

Кросс-валидация Accuracy: 84.00% (разброс: +/- 0.51%)
Кросс-валидация ROC AUC:  91.68%
